In [ ]:
import os
import glob
import cv2
import easyocr
import numpy as np
import editdistance
import matplotlib.pyplot as plt



# 2. Inicialitzem EasyOCR
# Posem 'en' (anglès) perquè inclou el model alfanumèric estàndard
print("Carregant el model d'EasyOCR...")
reader = easyocr.Reader(['en'], gpu=False) # Posa gpu=False si no tens gràfica Nvidia
print("Model carregat correctament.")

In [ ]:
def extreure_i_redrecar_matricula(imatge_path_o_array):
    """
    Detecta la matrícula a la imatge, en calcula la inclinació 
    i retorna un retall recte (de-skewed) de la mateixa.
    
    Retorna: np.ndarray (Imatge BGR de la matrícula) o None si falla.
    """
    # Accepta tant una ruta de fitxer com un array de OpenCV
    if isinstance(imatge_path_o_array, str):
        img = cv2.imread(imatge_path_o_array)
        if img is None: 
            print("PLATE_READER -- No s'ha trobat cap imatge")
            return None
    else:
        img = imatge_path_o_array.copy()

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    alçada_img = img.shape[0]

    # 1. Adaptive Threshold
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, 
        cv2.THRESH_BINARY_INV, 31, 15
    )

    # 2. Detecció de Blobs (Contorns)
    cnts, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    print(f"PLATE_DETECTOR -- {len(cnts)} contorns trobats")
    blobs_filtrats = []

    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        aspect_ratio = w / float(h)
        # Filtre de mida bàsica i proporció
        if (alçada_img * 0.02 < h < alçada_img * 0.8) and (0.15 < aspect_ratio < 1.5):
            blobs_filtrats.append((x, y, w, h))

    # 3. Agrupació per alineació
    blobs_filtrats.sort(key=lambda b: b[0])
    print(f"PLATE_DETECTOR -- {len(blobs_filtrats)} Blobs filtrats trobats")
    grups_horitzontals = []

    for i in range(len(blobs_filtrats)):
        x1, y1, w1, h1 = blobs_filtrats[i]
        grup = [(x1, y1, w1, h1)]
        
        for j in range(i + 1, len(blobs_filtrats)):
            x2, y2, w2, h2 = blobs_filtrats[j]
            
            dif_y = abs((y1 + h1/2) - (y2 + h2/2))
            dif_h = abs(h1 - h2)
            dist_x = x2 - (x1 + w1)
            
            # Criteris d'agrupació: Horitzontal, Altura similar i Distància raonable
            if dif_y < h1 * 0.3 and dif_h < h1 * 0.2 and 0 <= dist_x < h1 * 1.5:
                grup.append((x2, y2, w2, h2))
                x1, y1, w1, h1 = x2, y2, w2, h2 
                
        if len(grup) >= 3: # Una matrícula sol tenir 4 o més caràcters clars
            grups_horitzontals.append(grup)

    if not grups_horitzontals:
        print("PLATE_DETECTOR -- No s'han trobat matricules")
        return None

    # 4. Selecció i càlcul del Bounding Box de la Matrícula
    millor_grup = max(grups_horitzontals, key=len)
    xmin = min([b[0] for b in millor_grup])
    ymin = min([b[1] for b in millor_grup])
    xmax = max([b[0]+b[2] for b in millor_grup])
    ymax = max([b[1]+b[3] for b in millor_grup])

    marge = 10
    
    # 5. Rotació (De-skew) basada en els caràcters
    primer_char = millor_grup[0]
    ultim_char = millor_grup[-1]

    cx1 = primer_char[0] + (primer_char[2] / 2.0)
    cy1 = primer_char[1] + (primer_char[3] / 2.0)
    cx2 = ultim_char[0] + (ultim_char[2] / 2.0)
    cy2 = ultim_char[1] + (ultim_char[3] / 2.0)

    dy = cy2 - cy1
    dx = cx2 - cx1
    degrees = np.degrees(np.arctan2(dy, dx))

    # Retallem amb marge
    plate_region = img[max(0, ymin-marge) : min(img.shape[0], ymax+marge), 
                       max(0, xmin-marge) : min(img.shape[1], xmax+marge)].copy()

    # Apliquem el gir
    h, w = plate_region.shape[:2]
    cX, cY = w // 2, h // 2
    M = cv2.getRotationMatrix2D((cX, cY), degrees, 1.0)
    
    rotated_plate = cv2.warpAffine(plate_region, M, (w, h), borderValue=(180, 180, 180))

    return rotated_plate

In [ ]:
from ..src.detector import detect

In [ ]:
def segmentar_caracters_cnn(plate_img, mida_output=(28, 28)):
    """
    Donada la imatge d'una matrícula ja recta, n'aïlla cada lletra.
    Retorna: list[np.ndarray] (Una llista d'imatges binaritzades llestes per a una CNN)
    """
    if plate_img is None:
        return []

    gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)

    # 1. Binarització
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, 
        cv2.THRESH_BINARY_INV, 31, 15
    )

    # 2. Detecció de components de text
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = [cv2.boundingRect(c) for c in cnts]

    # 3. Càlcul de la mediana per ignorar marges i cargols
    altures_valides = [h for (x, y, w, h) in bboxes if h > int(plate_img.shape[0] * 0.2)]
    
    if not altures_valides:
        return []
        
    h_mediana = np.median(altures_valides)
    chars_filtrats = []

    # 4. Filtratge estricte i recol·lecció
    for (x, y, w, h) in bboxes:
        aspect_ratio = w / float(h)
        # Ha de tenir gairebé la mateixa alçada que la mitjana de les lletres
        if (h_mediana * 0.85 < h < h_mediana * 1.15) and (0.15 < aspect_ratio < 0.95):
            chars_filtrats.append((x, y, w, h))

    # Ordenem les lletres per x (d'esquerra a dreta)
    chars_filtrats.sort(key=lambda b: b[0])

    caracters_processats = []
    
    # 5. Retall i Resize (Output List)
    for (x, y, w, h) in chars_filtrats:
        # Retallem des de la imatge binaritzada directament!
        char_crop = thresh[y:y+h, x:x+w] 
        char_resized = cv2.resize(char_crop, mida_output, interpolation=cv2.INTER_AREA)
        
        caracters_processats.append(char_resized)

    return caracters_processats

In [ ]:
ruta_raw = "../data/raw/"
# Busquem totes les imatges jpg
imatges = sorted(glob.glob(os.path.join(ruta_raw, "*.jpg")))

exact_matches = 0
total_cer_distance = 0
total_caracters = 0
resultats = [] # Per guardar l'històric i analitzar errors

print(f"S'han trobat {len(imatges)} imatges a avaluar.\n")

for path_img in imatges:
    nom_base = os.path.splitext(os.path.basename(path_img))[0]
    path_txt = os.path.join(ruta_raw, nom_base + ".txt")
    
    # Llegim el Ground Truth (Text real)
    if not os.path.exists(path_txt):
        print(f"⚠️ Avís: No s'ha trobat el TXT per a {nom_base}. Ometent...")
        continue
        
    with open(path_txt, 'r', encoding='utf-8') as f:
        # Llegim la línia sencera i netegem els salts de línia del final
        linia = f.read().strip()
        
        # Separem pels tabuladors (\t)
        parts = linia.split('\t')
        
        # El text de la matrícula sempre és l'últim element (parts[-1])
        # Aprofitem per treure espais interns si n'hi ha i passar a majúscules
        ground_truth = parts[-1].replace(" ", "").upper()
        
    # ==========================================
    # PIPELINE D'EXECUCIÓ
    # ==========================================
    try:
        # 1. Cridem la funció del Notebook 1
        # Assumim que vas fer una funció que retorna la imatge de la placa redreçada
        plate_img = extreure_i_redrecar_matricula(path_img) 
        
        if plate_img is None:
            prediccio = "" # No s'ha trobat cap matrícula
            print("PIPELINE -- No s'ha trobat cap predicció")
        else:
            # 2. EasyOCR actua! (Substitueix el NB2 i NB3)
            # allowlist: limitem perquè no llegeixi símbols estranys o minúscules
            llista_mes_permesa = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
            
            # Executem EasyOCR directament sobre la placa retallada
            deteccions = reader.readtext(plate_img, detail=0, allowlist=llista_mes_permesa)
            
            # Unim totes les paraules detectades en un sol string net
            prediccio = "".join(deteccions).replace(" ", "").upper()
            
    except Exception as e:
        print(f"Error processant {nom_base}: {e}")
        prediccio = ""
        
    # ==========================================
    # CÀLCUL DE MÈTRIQUES PER AQUESTA IMATGE
    # ==========================================
    
    # 1. És un encert perfecte?
    es_encert = (prediccio == ground_truth)
    if es_encert:
        exact_matches += 1
        
    # 2. Calculem l'error a nivell de caràcter (CER)
    distancia = editdistance.eval(prediccio, ground_truth)
    total_cer_distance += distancia
    total_caracters += len(ground_truth)
    
    # Guardem per fer debugging després
    resultats.append({
        'imatge': nom_base,
        'real': ground_truth,
        'predit': prediccio,
        'distancia': distancia
    })
    
    # Print per veure com avança (opcional)
    estat = "✅" if es_encert else f"❌ (Dist: {distancia})"
    print(f"[{nom_base}] Real: {ground_truth} | Predit: {prediccio} {estat}")

print("\nProcés d'avaluació finalitzat!")

In [ ]:
# Càlcul de les mètriques finals
total_imatges = len(resultats)

if total_imatges > 0:
    accuracy = (exact_matches / total_imatges) * 100
    cer = (total_cer_distance / total_caracters) * 100

    print("=" * 40)
    print("📊 RESULTATS FINALS (KPIs)")
    print("=" * 40)
    print(f"Total processades: {total_imatges}")
    print(f"🎯 Exact Match Accuracy: {accuracy:.2f}% ({exact_matches}/{total_imatges})")
    print(f"📉 Character Error Rate (CER): {cer:.2f}%")
    print("=" * 40)
else:
    print("No hi havia imatges vàlides per avaluar.")

In [ ]:
import pandas as pd

# Creem un DataFrame per analitzar fàcilment on estem fallant
df_resultats = pd.DataFrame(resultats)

# Filtrem només els que han fallat
errors = df_resultats[df_resultats['distancia'] > 0].sort_values(by='distancia', ascending=False)

print(f"\nS'han detectat errors en {len(errors)} matrícules.")
if not errors.empty:
    print("Top 5 pitjors prediccions:")
    print(errors.head(5))